# BUS 104: Integer Linear Programming (ILP) — Homework Starter Notebook

Welcome to the **BUS 104 Integer Linear Programming Homework Starter Notebook** (`Homework_Files/ILP_Homework_Starter.ipynb` in the official **BUS 104 GitHub classroom repository**: <https://github.com/dxt48/BUS-104>).

### How to Use This Notebook for Your Canvas Homework
1. Every question on the **25-question Canvas Homework** references one of the **10 baseline models (`Problem 1` – `Problem 10`)** in this notebook (`Homework_Files/ILP_Homework_Starter.ipynb`).
2. Run each cell below using **PuLP 4.0** (`prob.add_variable(...)`, `prob.add_variable_dicts(...)`, `prob.solve(lp.COIN_CMD(msg=False))`) or replicate the tables in **Google Sheets with `OpenSolver`**.
3. Several Canvas questions will ask you to **modify parameters, add or drop logical/set-covering/subtour constraints, fix buggy Big-$M$ constants, or test contingency scenarios** directly in these cells and report the updated solver results.


---
## Problem 1: Precision Manufacturing Pure ILP & Branch-and-Bound (`P1_Precision_Manufacturing_ILP`)
A precision aerospace shop manufactures three titanium components ($x_1, x_2, x_3 \in \mathbb{Z}_+$) across three departments (`Milling`, `Lathe`, and `Laser`).
- **Objective:** $\max Z = 23x_1 + 17x_2 + 31x_3$ (in $\$\text{k}$)
- **Constraints:**
  - `Milling`: $4x_1 + 3x_2 + 5x_3 \le 46$ hours
  - `Lathe`: $2x_1 + 5x_2 + 3x_3 \le 41$ hours
  - `Laser`: $3x_1 + 2x_2 + 6x_3 \le 49$ hours


In [1]:
import pulp as lp

# 1. Continuous LP Relaxation (Root Node of Branch-and-Bound)
m1_lp = lp.LpProblem("P1_LP_Relaxation", lp.LpMaximize)
x_lp = m1_lp.add_variable_dicts("x", [1, 2, 3], lowBound=0, cat=lp.LpContinuous)
m1_lp += 23 * x_lp[1] + 17 * x_lp[2] + 31 * x_lp[3]
m1_lp += 4 * x_lp[1] + 3 * x_lp[2] + 5 * x_lp[3] <= 46, "Milling"
m1_lp += 2 * x_lp[1] + 5 * x_lp[2] + 3 * x_lp[3] <= 41, "Lathe"
m1_lp += 3 * x_lp[1] + 2 * x_lp[2] + 6 * x_lp[3] <= 49, "Laser"
m1_lp.solve(lp.COIN_CMD(msg=False))

print(f"LP Relaxation Obj: {lp.value(m1_lp.objective):.2f}")
print("LP Relaxation x  :", {i: round(lp.value(x_lp[i]), 4) for i in [1, 2, 3]})

# 2. Exact Pure ILP
m1_ilp = lp.LpProblem("P1_Precision_Manufacturing_ILP", lp.LpMaximize)
x_ilp = m1_ilp.add_variable_dicts("x", [1, 2, 3], lowBound=0, cat=lp.LpInteger)
m1_ilp += 23 * x_ilp[1] + 17 * x_ilp[2] + 31 * x_ilp[3]
m1_ilp += 4 * x_ilp[1] + 3 * x_ilp[2] + 5 * x_ilp[3] <= 46, "Milling"
m1_ilp += 2 * x_ilp[1] + 5 * x_ilp[2] + 3 * x_ilp[3] <= 41, "Lathe"
m1_ilp += 3 * x_ilp[1] + 2 * x_ilp[2] + 6 * x_ilp[3] <= 49, "Laser"
m1_ilp.solve(lp.COIN_CMD(msg=False))

print(f"Exact Pure ILP Obj: {lp.value(m1_ilp.objective):.2f}")
print("Exact Pure ILP x  :", {i: int(round(lp.value(x_ilp[i]))) for i in [1, 2, 3]})

# 3. Branch-and-Bound Children on x_3 (Node 1: x_3 <= 6, Node 2: x_3 >= 7)
for label, bound_type, rhs in [("Node_1_x3_le_6", "<=", 6), ("Node_2_x3_ge_7", ">=", 7)]:
    mb = lp.LpProblem(label, lp.LpMaximize)
    xb = mb.add_variable_dicts("x", [1, 2, 3], lowBound=0, cat=lp.LpContinuous)
    mb += 23 * xb[1] + 17 * xb[2] + 31 * xb[3]
    mb += 4 * xb[1] + 3 * xb[2] + 5 * xb[3] <= 46
    mb += 2 * xb[1] + 5 * xb[2] + 3 * xb[3] <= 41
    mb += 3 * xb[1] + 2 * xb[2] + 6 * xb[3] <= 49
    if bound_type == "<=":
        mb += xb[3] <= rhs
    else:
        mb += xb[3] >= rhs
    mb.solve(lp.COIN_CMD(msg=False))
    print(f"{label}: Obj = {lp.value(mb.objective):.2f}, x = {[round(lp.value(xb[i]), 3) for i in [1,2,3]]}")

LP Relaxation Obj: 279.00
LP Relaxation x  : {1: 3.4444, 2: 0.0, 3: 6.4444}
Exact Pure ILP Obj: 278.00
Exact Pure ILP x  : {1: 4, 2: 0, 3: 6}
Node_1_x3_le_6: Obj = 278.00, x = [4.0, 0.0, 6.0]
Node_2_x3_ge_7: Obj = 276.50, x = [0.0, 3.5, 7.0]


---
## Problem 2: Venture Capital Multi-Resource 0–1 Knapsack (`P2_Venture_Capital_Knapsack`)
A venture fund evaluates 8 deep-tech startups (`P1` through `P8`) under a **Capital Budget of $\$48\text{M}$** and an **Engineering Lead Pool of $32$ leads**.


In [2]:
import pulp as lp

projs = ["P1", "P2", "P3", "P4", "P5", "P6", "P7", "P8"]
npv = {"P1": 39, "P2": 34, "P3": 29, "P4": 26, "P5": 23, "P6": 19, "P7": 16, "P8": 12}
cap = {"P1": 19, "P2": 16, "P3": 13, "P4": 11, "P5": 10, "P6": 8,  "P7": 7,  "P8": 6}
eng = {"P1": 12, "P2": 10, "P3": 9,  "P4": 8,  "P5": 7,  "P6": 6,  "P7": 5,  "P8": 4}

CAP_BUDGET = 48
ENG_POOL = 32

m2 = lp.LpProblem("P2_Venture_Capital_Knapsack", lp.LpMaximize)
x2 = m2.add_variable_dicts("x", projs, cat=lp.LpBinary)
m2 += lp.lpSum(npv[p] * x2[p] for p in projs)
m2 += lp.lpSum(cap[p] * x2[p] for p in projs) <= CAP_BUDGET, "Capital_Cap"
m2 += lp.lpSum(eng[p] * x2[p] for p in projs) <= ENG_POOL, "Engineer_Cap"
m2.solve(lp.COIN_CMD(msg=False))

sel2 = [p for p in projs if lp.value(x2[p]) > 0.5]
print(f"Optimal 0-1 ILP NPV: ${lp.value(m2.objective):.0f}M | Selected: {sel2}")
print(f"Capital Used: ${sum(cap[p] for p in sel2)}M / ${CAP_BUDGET}M | Engineers Used: {sum(eng[p] for p in sel2)} / {ENG_POOL}")

# Compare with Naive Greedy Heuristic sorted by NPV-per-Capital ratio (npv[p] / cap[p])
greedy_order = sorted(projs, key=lambda p: npv[p] / cap[p], reverse=True)
g_sel, g_cap, g_eng = [], 0, 0
for p in greedy_order:
    if g_cap + cap[p] <= CAP_BUDGET and g_eng + eng[p] <= ENG_POOL:
        g_sel.append(p)
        g_cap += cap[p]
        g_eng += eng[p]
print("Greedy Order (NPV/Cap):", [(p, round(npv[p]/cap[p], 3)) for p in greedy_order])
print(f"Greedy Heuristic NPV: ${sum(npv[p] for p in g_sel)}M | Selected: {g_sel} (Cap={g_cap}, Eng={g_eng})")

Optimal 0-1 ILP NPV: $105M | Selected: ['P2', 'P3', 'P4', 'P7']
Capital Used: $47M / $48M | Engineers Used: 32 / 32
Greedy Order (NPV/Cap): [('P6', 2.375), ('P4', 2.364), ('P5', 2.3), ('P7', 2.286), ('P3', 2.231), ('P2', 2.125), ('P1', 2.053), ('P8', 2.0)]
Greedy Heuristic NPV: $96M | Selected: ['P6', 'P4', 'P5', 'P7', 'P8'] (Cap=42, Eng=30)


---
## Problem 3: Cloud Architecture 0–1 Knapsack with Logical Dependencies (`P3_Cloud_Architecture_Logic_ILP`)
A cloud architect selects from 8 candidate microservices (`M1` through `M8`) under a **Cloud Budget of $\$72\text{k}$** subject to 5 architectural dependency rules:
1. **Rule 1 (Prerequisite):** `M3` requires `M1` ($x_{M3} - x_{M1} \le 0$)
2. **Rule 2 (Mutual Exclusion):** `M2` and `M5` cannot both be deployed ($x_{M2} + x_{M5} \le 1$)
3. **Rule 3 (Exact Choice):** Exactly one of `M4` or `M6` must be deployed ($x_{M4} + x_{M6} = 1$)
4. **Rule 4 (Joint Prerequisite):** `M7` requires BOTH `M1` AND `M4` ($x_{M7} - x_{M1} \le 0$ and $x_{M7} - x_{M4} \le 0$)
5. **Rule 5 (Cluster Limit):** At most 2 of `{M1, M3, M7}` can run simultaneously ($x_{M1} + x_{M3} + x_{M7} \le 2$)


In [3]:
import pulp as lp

ms = ["M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8"]
val  = {"M1": 28, "M2": 40, "M3": 35, "M4": 24, "M5": 37, "M6": 22, "M7": 33, "M8": 25}
cost = {"M1": 15, "M2": 20, "M3": 17, "M4": 13, "M5": 18, "M6": 12, "M7": 16, "M8": 11}
BUDGET_P3 = 72

# Base Knapsack (No Logical Constraints)
m3_base = lp.LpProblem("P3_Base_Knapsack", lp.LpMaximize)
x3_b = m3_base.add_variable_dicts("x", ms, cat=lp.LpBinary)
m3_base += lp.lpSum(val[i] * x3_b[i] for i in ms)
m3_base += lp.lpSum(cost[i] * x3_b[i] for i in ms) <= BUDGET_P3
m3_base.solve(lp.COIN_CMD(msg=False))
sel3_b = [i for i in ms if lp.value(x3_b[i]) > 0.5]
print(f"Base Knapsack (No Logic): Value = ${lp.value(m3_base.objective):.0f}k | Selected = {sel3_b} | Cost = ${sum(cost[i] for i in sel3_b)}k")

# Full Logic-Constrained ILP
m3 = lp.LpProblem("P3_Cloud_Architecture_Logic_ILP", lp.LpMaximize)
x3 = m3.add_variable_dicts("x", ms, cat=lp.LpBinary)
m3 += lp.lpSum(val[i] * x3[i] for i in ms)
m3 += lp.lpSum(cost[i] * x3[i] for i in ms) <= BUDGET_P3, "Budget"
m3 += x3["M3"] - x3["M1"] <= 0, "Rule1_Prereq"
m3 += x3["M2"] + x3["M5"] <= 1, "Rule2_Mutex"
m3 += x3["M4"] + x3["M6"] == 1, "Rule3_ExactOne"
m3 += x3["M7"] - x3["M1"] <= 0, "Rule4a_JointPrereq1"
m3 += x3["M7"] - x3["M4"] <= 0, "Rule4b_JointPrereq2"
m3 += x3["M1"] + x3["M3"] + x3["M7"] <= 2, "Rule5_ClusterCap"
m3.solve(lp.COIN_CMD(msg=False))
sel3 = [i for i in ms if lp.value(x3[i]) > 0.5]
print(f"Full Logic ILP          : Value = ${lp.value(m3.objective):.0f}k | Selected = {sel3} | Cost = ${sum(cost[i] for i in sel3)}k")

Base Knapsack (No Logic): Value = $145k | Selected = ['M1', 'M5', 'M6', 'M7', 'M8'] | Cost = $72k
Full Logic ILP          : Value = $127k | Selected = ['M1', 'M2', 'M3', 'M4'] | Cost = $65k


---
## Problem 4: Biotech Reactor Fixed-Charge MILP & Big-$M$ Calibration (`P4_Biotech_Fixed_Charge_MILP`)
A biomanufacturing plant produces 4 specialty biologics ($x_1, x_2, x_3, x_4 \ge 0$, continuous batches) across 3 sterile reactors ($y_1, y_2, y_3 \in \{0,1\}$):
- **Reactor 1** (Fixed sterilization setup cost $F_1 = \$190\text{k}$) hosts compounds $x_1$ and $x_2$: $x_1 + x_2 - M_1 y_1 \le 0$.
- **Reactor 2** (Fixed setup cost $F_2 = \$150\text{k}$) hosts compound $x_3$: $x_3 - M_2 y_2 \le 0$.
- **Reactor 3** (Fixed setup cost $F_3 = \$220\text{k}$) hosts compound $x_4$: $x_4 - M_3 y_3 \le 0$.
- **Resource Capacities:**
  - `Lab_Hours`: $2x_1 + 3x_2 + 2x_3 + 3x_4 \le 96$
  - `Purification`: $3x_1 + 1x_2 + 4x_3 + 2x_4 \le 90$


In [4]:
import pulp as lp

def solve_p4(M1_val=48.0, F1_val=190.0):
    m4 = lp.LpProblem("P4_Biotech_Fixed_Charge_MILP", lp.LpMaximize)
    x = m4.add_variable_dicts("x", [1, 2, 3, 4], lowBound=0, cat=lp.LpContinuous)
    y = m4.add_variable_dicts("y", [1, 2, 3], cat=lp.LpBinary)

    m4 += 26*x[1] + 32*x[2] + 38*x[3] + 29*x[4] - F1_val*y[1] - 150*y[2] - 220*y[3]
    m4 += 2*x[1] + 3*x[2] + 2*x[3] + 3*x[4] <= 96, "Lab_Hours"
    m4 += 3*x[1] + 1*x[2] + 4*x[3] + 2*x[4] <= 90, "Purification"

    # Big-M linking constraints
    m4 += x[1] + x[2] - M1_val * y[1] <= 0, "Link_R1"
    m4 += x[3] - 22.5 * y[2] <= 0, "Link_R2"
    m4 += x[4] - 32.0 * y[3] <= 0, "Link_R3"

    m4.solve(lp.COIN_CMD(msg=False))
    x_sol = {i: round(lp.value(x[i]), 2) for i in [1, 2, 3, 4]}
    y_sol = {j: int(round(lp.value(y[j]))) for j in [1, 2, 3]}
    gross = 26*x_sol[1] + 32*x_sol[2] + 38*x_sol[3] + 29*x_sol[4]
    print(f"[M1={M1_val}, F1={F1_val}] Net Profit = ${lp.value(m4.objective):.2f}k | Gross = ${gross:.2f}k | x = {x_sol} | y = {y_sol}")

solve_p4(M1_val=48.0, F1_val=190.0)  # Valid tight Big-M1 = 96/2 = 48
solve_p4(M1_val=15.0, F1_val=190.0)  # Buggy too-small Big-M1 = 15

[M1=48.0, F1=190.0] Net Profit = $974.00k | Gross = $1314.00k | x = {1: 0.0, 2: 20.4, 3: 17.4, 4: 0.0} | y = {1: 1, 2: 1, 3: 0}
[M1=15.0, F1=190.0] Net Profit = $852.50k | Gross = $1192.50k | x = {1: 0.0, 2: 15.0, 3: 18.75, 4: 0.0} | y = {1: 1, 2: 1, 3: 0}


---
## Problem 5: Semiconductor Foundry MOQ & Batch Envelope MILP (`P5_Semiconductor_MOQ_MILP`)
A specialty semiconductor foundry earns a guaranteed **$\$300\text{k}$ baseline retainer** from legacy contracts and decides whether to activate its Next-Gen EUV line ($y \in \{0,1\}$):
- If $y = 1$, the foundry pays a **$\$150\text{k}$ EUV mask activation fee** ($F = 150$) and its total EUV wafer output $W = x_1 + x_2 + x_3$ must satisfy a **Minimum Order Quantity (MOQ) of $L = 20$ batches** and a **Maximum Chamber Envelope of $U = 36$ batches**:
  $$20y \le x_1 + x_2 + x_3 \le 36y$$
- **Resource Capacities:**
  - `Cleanroom`: $1x_1 + 2x_2 + 1x_3 \le 48$
  - `Lithography`: $2x_1 + 1x_2 + 2x_3 \le 54$


In [5]:
import pulp as lp

def solve_p5(F_fee=150.0, L_moq=20.0, U_cap=36.0):
    m5 = lp.LpProblem("P5_Semiconductor_MOQ_MILP", lp.LpMaximize)
    x = m5.add_variable_dicts("x", [1, 2, 3], lowBound=0, cat=lp.LpContinuous)
    y = m5.add_variable("y", cat=lp.LpBinary)

    m5 += 300 + 26*x[1] + 22*x[2] + 19*x[3] - F_fee*y
    m5 += 1*x[1] + 2*x[2] + 1*x[3] <= 48, "Cleanroom"
    m5 += 2*x[1] + 1*x[2] + 2*x[3] <= 54, "Lithography"
    m5 += x[1] + x[2] + x[3] - L_moq * y >= 0, "MOQ_Lower"
    m5 += x[1] + x[2] + x[3] - U_cap * y <= 0, "Max_Envelope"

    m5.solve(lp.COIN_CMD(msg=False))
    x_sol = {i: round(lp.value(x[i]), 2) for i in [1, 2, 3]}
    w_tot = sum(x_sol.values())
    print(f"[F={F_fee}, L={L_moq}, U={U_cap}] Total Profit = ${lp.value(m5.objective):.2f}k | y = {int(lp.value(y))} | x = {x_sol} | Total Wafers = {w_tot:.1f}")

solve_p5(F_fee=150.0, L_moq=20.0, U_cap=36.0)

[F=150.0, L=20.0, U=36.0] Total Profit = $978.00k | y = 1 | x = {1: 20.0, 2: 14.0, 3: 0.0} | Total Wafers = 34.0


---
## Problem 6: Global Cold-Chain Capacitated Facility Location & Transshipment (`P6_Global_ColdChain_Transshipment`)
A pharmaceutical logistics network ships temperature-controlled biologics from **4 candidate regional warehouses (`W1`–`W4`, capacity $S_i = 70$ pallets each)** through **2 cross-docking hubs (`T1`, `T2`)** to **3 hospital demand zones (`D1`=55, `D2`=45, `D3`=35; total demand $= 135$ pallets)**.


In [6]:
import pulp as lp

W = ["W1", "W2", "W3", "W4"]
T = ["T1", "T2"]
D = ["D1", "D2", "D3"]

S_cap = {"W1": 70, "W2": 70, "W3": 70, "W4": 70}
F_fix = {"W1": 180, "W2": 130, "W3": 110, "W4": 90}
c_WT = {
    ("W1", "T1"): 2, ("W1", "T2"): 6,
    ("W2", "T1"): 4, ("W2", "T2"): 3,
    ("W3", "T1"): 5, ("W3", "T2"): 4,
    ("W4", "T1"): 6, ("W4", "T2"): 5,
}
c_TD = {
    ("T1", "D1"): 3, ("T1", "D2"): 4, ("T1", "D3"): 7,
    ("T2", "D1"): 6, ("T2", "D2"): 3, ("T2", "D3"): 2,
}
dem = {"D1": 55, "D2": 45, "D3": 35}

def solve_p6(force_close_w1=False):
    m6 = lp.LpProblem("P6_Global_ColdChain_Transshipment", lp.LpMinimize)
    y = m6.add_variable_dicts("y", W, cat=lp.LpBinary)
    f1 = m6.add_variable_dicts("f1", [(i, t) for i in W for t in T], lowBound=0)
    f2 = m6.add_variable_dicts("f2", [(t, d) for t in T for d in D], lowBound=0)

    fixed_expr = lp.lpSum(F_fix[i] * y[i] for i in W)
    var_expr = lp.lpSum(c_WT[i, t] * f1[i, t] for i in W for t in T) + lp.lpSum(c_TD[t, d] * f2[t, d] for t in T for d in D)
    m6 += fixed_expr + var_expr

    for i in W:
        m6 += lp.lpSum(f1[i, t] for t in T) - S_cap[i] * y[i] <= 0, f"Cap_{i}"
    for t in T:
        m6 += lp.lpSum(f1[i, t] for i in W) - lp.lpSum(f2[t, d] for d in D) == 0, f"Bal_{t}"
    for d in D:
        m6 += lp.lpSum(f2[t, d] for t in T) == dem[d], f"Dem_{d}"

    if force_close_w1:
        m6 += y["W1"] == 0, "Force_Close_W1"

    m6.solve(lp.COIN_CMD(msg=False))
    open_w = [i for i in W if lp.value(y[i]) > 0.5]
    print(f"[Close W1={force_close_w1}] Total Cost = ${lp.value(m6.objective):.0f}k (Fixed=${lp.value(fixed_expr):.0f}k, Shipping=${lp.value(var_expr):.0f}k) | Open: {open_w}")
    for i in open_w:
        for t in T:
            if lp.value(f1[i, t]) > 0:
                print(f"   {i} -> {t}: {lp.value(f1[i, t]):.0f} pallets")
    for t in T:
        for d in D:
            if lp.value(f2[t, d]) > 0:
                print(f"   {t} -> {d}: {lp.value(f2[t, d]):.0f} pallets")

solve_p6(force_close_w1=False)
solve_p6(force_close_w1=True)

[Close W1=False] Total Cost = $1030k (Fixed=$310k, Shipping=$720k) | Open: ['W1', 'W2']
   W1 -> T1: 65 pallets
   W2 -> T2: 70 pallets
   T1 -> D1: 55 pallets
   T1 -> D2: 10 pallets
   T2 -> D2: 35 pallets
   T2 -> D3: 35 pallets
[Close W1=True] Total Cost = $1135k (Fixed=$240k, Shipping=$895k) | Open: ['W2', 'W3']
   W2 -> T2: 70 pallets
   W3 -> T1: 55 pallets
   W3 -> T2: 10 pallets
   T1 -> D1: 55 pallets
   T2 -> D2: 45 pallets
   T2 -> D3: 35 pallets


---
## Problem 7: Metro EV Fast-Charger Grid Set Covering (`P7_Metro_EV_Charger_Set_Covering`)
A city transportation authority places EV fast-charging stations across a **$3 \times 4$ metropolitan grid (12 sectors, numbered `1` to `12`)**:
```
Row 0: [ 1 ] [ 2 ] [ 3 ] [ 4 ]
Row 1: [ 5 ] [ 6 ] [ 7 ] [ 8 ]
Row 2: [ 9 ] [10 ] [11 ] [12 ]
```
A station in sector $j$ covers sector $j$ itself plus all orthogonally adjacent sectors ($\text{Manhattan distance } |r_i - r_j| + |c_i - c_j| \le 1$).


In [7]:
import pulp as lp

sectors = list(range(1, 13))
coords = {s: ((s - 1) // 4, (s - 1) % 4) for s in sectors}
adj = {
    i: [j for j in sectors if abs(coords[i][0] - coords[j][0]) + abs(coords[i][1] - coords[j][1]) <= 1]
    for i in sectors
}
cost_s = {1: 16, 2: 21, 3: 19, 4: 15, 5: 18, 6: 28, 7: 26, 8: 17, 9: 14, 10: 22, 11: 20, 12: 13}

def solve_p7(downtown_double=False, strict_equality=False):
    m7 = lp.LpProblem("P7_Metro_EV_Charger_Set_Covering", lp.LpMinimize)
    x = m7.add_variable_dicts("x", sectors, cat=lp.LpBinary)
    m7 += lp.lpSum(cost_s[j] * x[j] for j in sectors)

    for i in sectors:
        if strict_equality:
            m7 += lp.lpSum(x[j] for j in adj[i]) == 1, f"Exact_{i}"
        elif downtown_double and i in [6, 7]:
            m7 += lp.lpSum(x[j] for j in adj[i]) >= 2, f"DoubleCov_{i}"
        else:
            m7 += lp.lpSum(x[j] for j in adj[i]) >= 1, f"Cov_{i}"

    res = m7.solve(lp.COIN_CMD(msg=False))
    status = res.status.name
    if status == "Optimal":
        chosen = [j for j in sectors if lp.value(x[j]) > 0.5]
        print(f"[DoubleDowntown={downtown_double}, Equality={strict_equality}] Status={status} | Min Cost = ${lp.value(m7.objective):.0f}M | Stations = {chosen}")
    else:
        print(f"[DoubleDowntown={downtown_double}, Equality={strict_equality}] Status={status}")

solve_p7(downtown_double=False, strict_equality=False)
solve_p7(downtown_double=True,  strict_equality=False)
solve_p7(downtown_double=False, strict_equality=True)

[DoubleDowntown=False, Equality=False] Status=Optimal | Min Cost = $64M | Stations = [3, 5, 9, 12]
[DoubleDowntown=True, Equality=False] Status=Optimal | Min Cost = $76M | Stations = [3, 5, 8, 10]
[DoubleDowntown=False, Equality=True] Status=Infeasible


---
## Problem 8: Autonomous Drone 7-Stop TSP & Subtour Elimination (`P8_Drone_Delivery_TSP_Subtours`)
An autonomous medical delivery drone visits **7 hospital clinics (`Nodes 1..7`)**:
- **North Cluster:** `{1, 2, 3}` (tight local triangle)
- **South Cluster:** `{4, 5, 6, 7}` (two tight local pairs `{4,5}` and `{6,7}`)


In [8]:
import pulp as lp
from itertools import combinations

nodes = [1, 2, 3, 4, 5, 6, 7]
dist8 = {
    (1,2):4,  (2,1):5,  (2,3):4,  (3,2):5,  (3,1):4,  (1,3):5,
    (4,5):3,  (5,4):3,  (6,7):3,  (7,6):3,
    (4,6):9,  (6,4):9,  (5,6):8,  (6,5):8,  (5,7):9,  (7,5):9,  (4,7):8,  (7,4):8,
    (2,4):14, (4,2):14, (3,5):15, (5,3):15, (3,7):13, (7,3):13, (1,6):18, (6,1):18,
    (1,4):17, (4,1):17, (2,6):17, (6,2):17
}
for i in nodes:
    for j in nodes:
        if i != j and (i, j) not in dist8:
            dist8[(i, j)] = 40

def extract_cycles(x_var):
    succ = {i: [j for j in nodes if i != j and lp.value(x_var[i, j]) > 0.5][0] for i in nodes}
    visited, cycles = set(), []
    for start in nodes:
        if start not in visited:
            cyc, cur = [], start
            while cur not in visited:
                visited.add(cur)
                cyc.append(cur)
                cur = succ[cur]
            cyc.append(cur)
            cycles.append(cyc)
    return cycles

def solve_p8(subtour_sizes=()):
    m8 = lp.LpProblem("P8_Drone_Delivery_TSP", lp.LpMinimize)
    x = m8.add_variable_dicts("x", [(i, j) for i in nodes for j in nodes if i != j], cat=lp.LpBinary)
    m8 += lp.lpSum(dist8[i, j] * x[i, j] for i in nodes for j in nodes if i != j)
    for i in nodes:
        m8 += lp.lpSum(x[i, j] for j in nodes if j != i) == 1, f"Out_{i}"
        m8 += lp.lpSum(x[j, i] for j in nodes if j != i) == 1, f"In_{i}"
    for k in subtour_sizes:
        for S in combinations(nodes, k):
            m8 += lp.lpSum(x[i, j] for i in S for j in S if i != j) <= k - 1
    m8.solve(lp.COIN_CMD(msg=False))
    print(f"Subtour sizes eliminated {list(subtour_sizes)}: Time = {lp.value(m8.objective):.0f} min | Cycles = {extract_cycles(x)}")

solve_p8(subtour_sizes=())               # Degree constraints only (Subtour Trap!)
solve_p8(subtour_sizes=(2,))             # Eliminating ONLY 2-node subtours
solve_p8(subtour_sizes=(2, 3))           # Eliminating 2-node and 3-node subtours
solve_p8(subtour_sizes=(2, 3, 4, 5, 6))  # Full DFJ Subtour Elimination (Single Tour)

Subtour sizes eliminated []: Time = 24 min | Cycles = [[1, 2, 3, 1], [4, 5, 4], [6, 7, 6]]
Subtour sizes eliminated [2]: Time = 34 min | Cycles = [[1, 2, 3, 1], [4, 5, 6, 7, 4]]
Subtour sizes eliminated [2, 3]: Time = 49 min | Cycles = [[1, 2, 4, 5, 6, 7, 3, 1]]
Subtour sizes eliminated [2, 3, 4, 5, 6]: Time = 49 min | Cycles = [[1, 2, 4, 5, 6, 7, 3, 1]]


---
## Problem 9: NBA Analytics Lineup Optimization with Chemistry Rules (`P9_NBA_Analytics_Lineup_ILP`)
A front-office analytics team selects a **5-player starting unit** (`2 Guards`, `2 Forwards`, `1 Center`) from 10 available players (`P1`–`P10`) under a **Salary Cap of $\$53.0\text{k}$**:
- **Positional Quotas:**
  - Guards (`P1..P4`): $x_{P1} + x_{P2} + x_{P3} + x_{P4} = 2$
  - Forwards (`P5..P8`): $x_{P5} + x_{P6} + x_{P7} + x_{P8} = 2$
  - Centers (`P9..P10`): $x_{P9} + x_{P10} = 1$
- **4 Locker-Room & Tactical Chemistry Rules:**
  - **Rule 1 (Paint Spacing Clash):** `P6` and `P9` clog the paint together ($x_{P6} + x_{P9} \le 1$)
  - **Rule 2 (Pick-and-Roll Stacking):** Center `P10` requires Point Guard `P1` ($x_{P10} - x_{P1} \le 0$)
  - **Rule 3 (Ball-Dominant Usage Limit):** At most 2 of `{P1, P2, P5}` can start together ($x_{P1} + x_{P2} + x_{P5} \le 2$)
  - **Rule 4 (Luxury Apron Rule):** At most 2 of `{P1, P2, P10}` can start together ($x_{P1} + x_{P2} + x_{P10} \le 2$)


In [9]:
import pulp as lp

players = ["P1", "P2", "P3", "P4", "P5", "P6", "P7", "P8", "P9", "P10"]
pts = {"P1": 56, "P2": 49, "P3": 38, "P4": 30, "P5": 52, "P6": 45, "P7": 35, "P8": 28, "P9": 44, "P10": 41}
sal = {"P1": 13.0, "P2": 11.0, "P3": 8.5, "P4": 6.0, "P5": 12.0, "P6": 9.5, "P7": 7.5, "P8": 5.5, "P9": 9.0, "P10": 8.0}

def solve_p9(cap_limit=53.0, enforce_chemistry=True):
    m9 = lp.LpProblem("P9_NBA_Analytics_Lineup_ILP", lp.LpMaximize)
    x = m9.add_variable_dicts("x", players, cat=lp.LpBinary)
    m9 += lp.lpSum(pts[p] * x[p] for p in players)
    m9 += lp.lpSum(sal[p] * x[p] for p in players) <= cap_limit, "Salary_Cap"
    m9 += lp.lpSum(x[p] for p in ["P1", "P2", "P3", "P4"]) == 2, "Guards"
    m9 += lp.lpSum(x[p] for p in ["P5", "P6", "P7", "P8"]) == 2, "Forwards"
    m9 += lp.lpSum(x[p] for p in ["P9", "P10"]) == 1, "Centers"

    if enforce_chemistry:
        m9 += x["P6"] + x["P9"] <= 1, "Chem1_PaintClash"
        m9 += x["P10"] - x["P1"] <= 0, "Chem2_PnR_Stack"
        m9 += x["P1"] + x["P2"] + x["P5"] <= 2, "Chem3_UsageCap"
        m9 += x["P1"] + x["P2"] + x["P10"] <= 2, "Chem4_ApronRule"

    m9.solve(lp.COIN_CMD(msg=False))
    sel = [p for p in players if lp.value(x[p]) > 0.5]
    print(f"[Cap=${cap_limit}k, Chemistry={enforce_chemistry}] Projected Pts = {lp.value(m9.objective):.0f} | Salary = ${sum(sal[p] for p in sel):.1f}k | Lineup = {sel}")

solve_p9(cap_limit=53.0, enforce_chemistry=False)  # Naive All-Star Lineup
solve_p9(cap_limit=53.0, enforce_chemistry=True)   # Full Chemistry-Optimal Lineup
solve_p9(cap_limit=47.0, enforce_chemistry=True)   # Tighter Salary Cap ($47.0k)

[Cap=$53.0k, Chemistry=False] Projected Pts = 236 | Salary = $52.5k | Lineup = ['P1', 'P2', 'P5', 'P7', 'P9']
[Cap=$53.0k, Chemistry=True] Projected Pts = 232 | Salary = $51.0k | Lineup = ['P1', 'P3', 'P5', 'P6', 'P10']
[Cap=$47.0k, Chemistry=True] Projected Pts = 215 | Salary = $46.5k | Lineup = ['P1', 'P3', 'P6', 'P7', 'P10']


---
## Problem 10: Cross-Border FX & Crypto Currency Arbitrage Tour (`P10_Crypto_FX_Arbitrage_Tour`)
An algorithmic FX desk starts with **$\$250{,}000\text{ USD}$** and searches for the most profitable **directed currency arbitrage cycle** across 5 currencies (`USD`, `EUR`, `GBP`, `JPY`, `CHF`) and 9 quoted exchange-rate arcs.
- Using a logarithmic objective $\max Z = \sum_{(i,j) \in A} \ln\big(r_{ij}(1 - f)\big) x_{ij}$ linearizes the compounded tour multiplier $M = \prod_{(i,j)} \big(r_{ij}(1 - f)\big)^{x_{ij}} = e^Z$, where $f$ is the per-transaction exchange fee.


In [10]:
import math
import pulp as lp

curr = ["USD", "EUR", "GBP", "JPY", "CHF"]
fx10 = {
    ("USD", "EUR"): 0.93,
    ("USD", "GBP"): 0.79,
    ("EUR", "GBP"): 0.88,
    ("EUR", "CHF"): 0.985,
    ("GBP", "USD"): 1.26,
    ("GBP", "JPY"): 1.92,
    ("GBP", "CHF"): 1.12,
    ("JPY", "CHF"): 0.59,
    ("CHF", "USD"): 1.15,
}
CAPITAL_USD = 250_000

def solve_p10(fee=0.0):
    m10 = lp.LpProblem("P10_Crypto_FX_Arbitrage_Tour", lp.LpMaximize)
    x = m10.add_variable_dicts("x", list(fx10.keys()), cat=lp.LpBinary)

    m10 += lp.lpSum(math.log(fx10[i, j] * (1.0 - fee)) * x[i, j] for (i, j) in fx10)

    # Flow balance at every currency node
    for c in curr:
        inflow  = lp.lpSum(x[i, j] for (i, j) in fx10 if j == c)
        outflow = lp.lpSum(x[i, j] for (i, j) in fx10 if i == c)
        m10 += inflow - outflow == 0, f"Flow_{c}"
        m10 += outflow <= 1, f"Simple_{c}"

    # Tour must start and end at USD
    m10 += lp.lpSum(x["USD", j] for (i, j) in fx10 if i == "USD") == 1, "Start_USD"

    m10.solve(lp.COIN_CMD(msg=False))
    z_val = lp.value(m10.objective)
    mult = math.exp(z_val)
    profit = CAPITAL_USD * (mult - 1.0)
    arcs = [(i, j) for (i, j) in fx10 if lp.value(x[i, j]) > 0.5]
    print(f"[Fee={fee*100:.1f}%] Log-Obj Z = {z_val:.5f} | Multiplier = {mult:.6f} | Net Profit = ${profit:,.2f} | Arcs = {arcs}")

solve_p10(fee=0.00)  # Zero transaction fee (0.0%)
solve_p10(fee=0.01)  # 1.0% per-hop transaction fee

[Fee=0.0%] Log-Obj Z = 0.06405 | Multiplier = 1.066146 | Net Profit = $16,536.51 | Arcs = [('USD', 'EUR'), ('EUR', 'GBP'), ('GBP', 'JPY'), ('JPY', 'CHF'), ('CHF', 'USD')]
[Fee=1.0%] Log-Obj Z = 0.02193 | Multiplier = 1.022169 | Net Profit = $5,542.19 | Arcs = [('USD', 'EUR'), ('EUR', 'CHF'), ('CHF', 'USD')]
